# Hands-on AI in Healthcare - Chapter 3: Classical Statistics for Population Health

## 3.1 Setting Up Your Workbench

### 3.1.1 Installing and Importing Libraries

In [ ]:
# Install the libraries we'll use throughout the book.
# You only need to run this cell once per environment.
%pip install pandas matplotlib scikit-learn numpy

In [ ]:
import glob

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

print(f"pandas     : {pd.__version__}")
print(f"numpy      : {np.__version__}")
print(f"matplotlib : {plt.matplotlib.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print("\nAll libraries loaded successfully!")

### 3.1.2 About the Data

Your file structure should look like this:

```
chapter_3.ipynb
working/
    analytic_data2024.csv
```

In [ ]:
# Look for the analytic data CSV in the working/ folder.
# This pattern matches any file starting with "analytic_data" so it will
# work regardless of which year's file you downloaded.
csv_files = sorted(glob.glob("working/analytic_data*.csv"))

if not csv_files:
    raise FileNotFoundError(
        "No analytic_data*.csv file found in the working/ folder.\n"
        "Please download the file from countyhealthrankings.org and place it "
        "in a folder called 'working' next to this notebook."
    )

csv_path = csv_files[-1]  # Use the most recent file if multiple are present
print(f"Loading: {csv_path}")

# Read the CSV.
# Row 1, the second row, has computer-friendly variable names (our column names);
# row 0, or the first row, has descriptions that we skip.
# Using encoding="latin-1" handles special characters that occasionally appear in county names.
chr_data = pd.read_csv(
    csv_path,
    header=0,
    skiprows=[0],
    encoding="latin-1",
)

print(f"Dataset shape: {chr_data.shape[0]:,} rows × {chr_data.shape[1]} columns")

### [BONUS] Exploring the Data

In [ ]:
# Show the first few rows to get a feel for the data.
chr_data.head()

In [ ]:
# List all column names. This is especially useful with large datasets
# where .head() truncates the display.
print("Number of columns:", len(chr_data.columns))
print()
for col in chr_data.columns:
    print(col)

The County Health Rankings uses a coding scheme for its variable names. Each health
measure has a number, and the raw value for that measure is stored in a column named
`v{number}_rawvalue`. The columns we need are:

| Column | Description |
|--------|-------------|
| `statecode` | Two-digit state FIPS code |
| `state` | State abbreviation |
| `county` | County name |
| `v060_rawvalue` | Diabetes prevalence (proportion of adults 20+ with diagnosed diabetes) |
| `v063_rawvalue` | Median household income (dollars) |

Let's confirm these columns exist and see what the values look like.

In [ ]:
# The columns we'll work with.
DIABETES_COL = "v060_rawvalue"
INCOME_COL = "v063_rawvalue"

# Verify they exist in the dataset.
for col in [DIABETES_COL, INCOME_COL, "state", "county"]:
    if col in chr_data.columns:
        print(f"  ✓  '{col}' found")
    else:
        print(f"  ✗  '{col}' NOT found — check the column list above")

In [ ]:
# Quick summary of our two key columns.
chr_data[[DIABETES_COL, INCOME_COL]].describe()

### 3.1.4 Filtering to California

In [ ]:
# Choose a state. Change this abbreviation to explore a different state.
DIABETES_COL = "v060_rawvalue"
INCOME_COL = "v063_rawvalue"
STATE = "CA"

# Filter to the chosen state and drop rows where either measure is missing.
# The first row for each state is a state-level summary (countycode == 0),
# which we exclude so we only have individual counties.
state_data = (
    chr_data[
        (chr_data["state"] == STATE)
        & (chr_data["countycode"] != 0)
    ]
    .dropna(subset=[DIABETES_COL, INCOME_COL])
    .copy()
)

print(f"Counties in {STATE} with complete data: {len(state_data)}")

### 3.1.5 Your First Visualization: Diabetes vs. Income

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x_vals = state_data[INCOME_COL]
y_vals = state_data[DIABETES_COL] * 100  # Convert proportion to percentage
counties = state_data["county"]

ax.scatter(
    x_vals,
    y_vals,
    alpha=0.6,
    edgecolors="steelblue",
    facecolors="lightblue",
    linewidths=0.8,
)

# Label a few notable counties so the data feels real.
# We pick the extremes: highest/lowest diabetes, highest/lowest income,
# and any interesting outliers.
label_counties = set()
if len(x_vals) > 0:
    label_counties.add(y_vals.idxmax())  # highest diabetes
    label_counties.add(y_vals.idxmin())  # lowest diabetes
    label_counties.add(x_vals.idxmax())  # highest income
    label_counties.add(x_vals.idxmin())  # lowest income

for idx in label_counties:
    ax.annotate(
        counties.loc[idx],
        (x_vals.loc[idx], y_vals.loc[idx]),
        textcoords="offset points",
        xytext=(8, 4),
        fontsize=8,
        color="dimgray",
        arrowprops=dict(arrowstyle="-", color="gray", lw=0.5),
    )

ax.set_xlabel("Median Household Income ($)", fontsize=12)
ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
ax.set_title(
    f"Diabetes Prevalence vs. Median Household Income\n"
    f"California Counties",
    fontsize=14,
)

# Format the x-axis labels as dollar amounts.
ax.xaxis.set_major_formatter(
    plt.FuncFormatter(lambda x, _: f"${x:,.0f}")
)

ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### What Do You See?

If your environment is set up correctly, you should see a scatter plot with a clear
downward trend.

### Workbench Check

If you made it this far without errors, your environment is ready. Here's what we confirmed:

1. **pandas** can load data from a local CSV file
2. **matplotlib** can render charts in your notebook
3. **scikit-learn** is installed (we'll use it starting in Section 3.3)
4. You can load and filter the County Health Rankings dataset

---

## 3.2 Understanding the Data: What Does a Typical County Look Like?

In [ ]:
# Define the health indicators we'll study in this section.
INDICATORS = {
    "v060_rawvalue": "Diabetes Prevalence",
    "v009_rawvalue": "Adult Smoking Rate",
    "v011_rawvalue": "Adult Obesity Rate",
    "v005_rawvalue": "Preventable Hospital Stays",
    "v063_rawvalue": "Median Household Income",
}

# Verify all columns are present.
for col, label in INDICATORS.items():
    if col in chr_data.columns:
        print(f"  ✓  '{col}'  ({label})")
    else:
        print(f"  ✗  '{col}'  ({label}) — NOT found")

### 3.2.1 Summary Statistics

In [ ]:
# Build a clean dataframe of California county indicators.
ca_indicators = (
    state_data[list(INDICATORS.keys())]
    .rename(columns=INDICATORS)
    .copy()
)

# Scale proportion columns to percentages for readability.
# Preventable hospital stays is already a rate per 100,000, so leave it as-is.
# Median household income is in dollars, so leave it as-is.
pct_cols = ["Diabetes Prevalence", "Adult Smoking Rate", "Adult Obesity Rate"]
ca_indicators[pct_cols] = ca_indicators[pct_cols] * 100

ca_indicators.describe().round(2)

### [BONUS] Distributions: Histograms

A histogram bins the data into ranges and counts how many counties fall into each bin.
The shape of the histogram tells us about the distribution. A bell-shaped histogram
indicates a roughly normal (Gaussian) distribution. A histogram with a long tail on
one side indicates a skewed distribution.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
fig.suptitle("Distribution of Health Indicators — California Counties", fontsize=14)

plot_cols = ["Diabetes Prevalence", "Adult Smoking Rate",
             "Adult Obesity Rate", "Preventable Hospital Stays"]
colors = ["steelblue", "seagreen", "coral", "mediumpurple"]

for ax, col, color in zip(axes.flat, plot_cols, colors):
    data = ca_indicators[col].dropna()
    ax.hist(data, bins=15, color=color, edgecolor="white", alpha=0.8)
    ax.axvline(data.mean(), color="black", linestyle="--", linewidth=1.2,
               label=f"Mean: {data.mean():.1f}")
    ax.axvline(data.median(), color="red", linestyle="-", linewidth=1.2,
               label=f"Median: {data.median():.1f}")
    ax.set_xlabel(col)
    ax.set_ylabel("Number of Counties")
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

Notice which distributions look roughly symmetric (bell-shaped) and which are skewed.
The dashed black line (mean) and solid red line (median) help you see the skew: when
the mean is pulled to the right of the median, the distribution has a right skew,
meaning a few counties with very high values are pulling the average up.

Understanding the shape of your data matters because many statistical techniques —
including ordinary linear regression — make assumptions about how the data is distributed.
When those assumptions are violated, the results can be misleading.

### 3.2.2 Correlation: Which Indicators Move Together?

In [ ]:
# Compute the correlation matrix.
corr_matrix = ca_indicators.corr().round(3)

# Display it as a heatmap.
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(corr_matrix, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

# Label the axes.
labels = corr_matrix.columns
ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=10)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=10)

# Annotate each cell with the correlation value.
for i in range(len(labels)):
    for j in range(len(labels)):
        ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                ha="center", va="center", fontsize=10,
                color="white" if abs(corr_matrix.iloc[i, j]) > 0.6 else "black")

plt.colorbar(im, ax=ax, label="Correlation", shrink=0.8)
ax.set_title("Correlation Between Health Indicators — California Counties", fontsize=13)
plt.tight_layout()
plt.show()

---

## 3.3 Linear Regression: Drawing the Line

In [ ]:
OBESITY_COL = "v011_rawvalue"

# Prepare the data. scikit-learn expects X as a 2D array.
X = state_data[[OBESITY_COL]].values * 100 # percentage
y = state_data[DIABETES_COL].values * 100  # percentage

# Fit a simple linear regression.
model_simple = LinearRegression()
model_simple.fit(X, y)

# Predictions for the regression line.
y_pred = model_simple.predict(X)

# Metrics.
mse = mean_squared_error(y, y_pred)
r2 = r2_score(y, y_pred)

print(f"Slope (m):     {model_simple.coef_[0]:.6f}")
print(f"Intercept (b): {model_simple.intercept_:.2f}")
print(f"MSE:           {mse:.4f}")
print(f"R²:            {r2:.4f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot of actual data.
ax.scatter(X, y, alpha=0.6, edgecolors="steelblue", facecolors="lightblue",
           linewidths=0.8, label="Actual")

# Regression line.
income_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ax.plot(income_range, model_simple.predict(income_range),
        color="red", linewidth=2, label=f"Regression line (R²={r2:.3f})")

ax.set_xlabel("Adult Obesity Rate (%)", fontsize=12)
ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
ax.set_title("Linear Regression: Diabetes Prevalence vs. Obesity\nCalifornia Counties",
             fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### [BONUS] Residuals: What the Model Misses

The vertical distance between each dot and the regression line is the **residual** — the
error for that county. Plotting the residuals helps us see patterns the model is missing.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

residuals = y - y_pred
ax.scatter(y_pred, residuals, alpha=0.6, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8)
ax.axhline(0, color="red", linewidth=1.5, linestyle="--")
ax.set_xlabel("Predicted Diabetes Prevalence (%)", fontsize=12)
ax.set_ylabel("Residual (Actual − Predicted)", fontsize=12)
ax.set_title("Residual Plot — Simple Linear Regression", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

If the linear model were a perfect fit, the residuals would be randomly scattered around
the zero line with no visible pattern. Any pattern in the residuals — a curve, a funnel
shape, clusters — tells us the model is missing something.

**Key bridge to neural networks:** This idea of "measure how wrong you are, then adjust"
is the fundamental principle behind how neural networks learn. A single linear regression
is, in a very real sense, a one-neuron neural network with no activation function. We'll
build on this idea throughout the chapter.

### 3.3.1 Using the Model to Make Predictions

In [ ]:
# Split: 70% for training, 30% for testing.
X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X, y, test_size=0.3, random_state=42
)
print(f"Training counties: {len(X_train_s)}")
print(f"Test counties:     {len(X_test_s)}")

# Fit the model on ONLY the training data.
model_split = LinearRegression()
model_split.fit(X_train_s, y_train_s)

# Predict on the test set — counties the model has never seen.
y_test_pred = model_split.predict(X_test_s)

r2_train_s = model_split.score(X_train_s, y_train_s)
r2_test_s = model_split.score(X_test_s, y_test_s)
mse_test_s = mean_squared_error(y_test_s, y_test_pred)

print(f"\nR² on training data: {r2_train_s:.4f}")
print(f"R² on test data:     {r2_test_s:.4f}")
print(f"MSE on test data:    {mse_test_s:.4f}")

In [ ]:
# Visualize: training data, test data, and the regression line.
fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(X_train_s, y_train_s, alpha=0.6, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8, label="Training data")
ax.scatter(X_test_s, y_test_s, alpha=0.8, edgecolors="coral",
           facecolors="lightsalmon", linewidths=0.8, marker="s", s=60,
           label="Test data")

# Draw the regression line (fit on training data only).
income_range = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
ax.plot(income_range, model_split.predict(income_range),
        color="red", linewidth=2, label=f"Regression line (test R²={r2_test_s:.3f})")

ax.set_xlabel("Adult Obesity Rate (%)", fontsize=12)
ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
ax.set_title("Train/Test Split: Can the Model Predict Unseen Counties?\nCalifornia",
             fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3.4 Multiple Regression: Health Is Never Just One Thing

In [ ]:
# Predictors: income, obesity, smoking, and preventable hospital stays.
PREDICTORS = {
    "v063_rawvalue": "Median Household Income",
    "v011_rawvalue": "Adult Obesity Rate",
    "v009_rawvalue": "Adult Smoking Rate",
    "v005_rawvalue": "Preventable Hospital Stays",
}

# Build a clean dataframe for California, dropping rows with any missing predictor.
multi_data = state_data[["county"] + list(PREDICTORS.keys()) + [DIABETES_COL]].dropna().copy()

X_multi = multi_data[list(PREDICTORS.keys())].values
y_multi = multi_data[DIABETES_COL].values * 100  # percentage

# Fit a multiple regression.
model_multi = LinearRegression()
model_multi.fit(X_multi, y_multi)

y_pred_multi = model_multi.predict(X_multi)
r2_multi = r2_score(y_multi, y_pred_multi)
mse_multi = mean_squared_error(y_multi, y_pred_multi)

print("Multiple Regression Results")
print("=" * 50)
for name, coef in zip(PREDICTORS.values(), model_multi.coef_):
    print(f"  {name:30s}  weight = {coef:+.6f}")
print(f"  {'Intercept':30s}  bias   = {model_multi.intercept_:+.4f}")
print(f"\n  MSE: {mse_multi:.4f}")
print(f"  R²:  {r2_multi:.4f}")
print(f"\nCompare to simple regression R²: {r2:.4f}")

In [ ]:
# Visualize: actual vs. predicted for the multiple regression model.
fig, ax = plt.subplots(figsize=(8, 6))

ax.scatter(y_multi, y_pred_multi, alpha=0.6, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8)

# Perfect-prediction line.
lims = [min(y_multi.min(), y_pred_multi.min()) - 0.5,
        max(y_multi.max(), y_pred_multi.max()) + 0.5]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")

ax.set_xlabel("Actual Diabetes Prevalence (%)", fontsize=12)
ax.set_ylabel("Predicted Diabetes Prevalence (%)", fontsize=12)
ax.set_title("Multiple Regression: Actual vs. Predicted\nCalifornia Counties",
             fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_aspect("equal")
ax.set_xlim(lims)
ax.set_ylim(lims)
plt.tight_layout()
plt.show()

---

## 3.5 When Lines Aren't Enough: Polynomial Regression

In [ ]:
# Prepare the data: preventable hospital stays vs. diabetes prevalence.
HOSP_COL = "v005_rawvalue"

poly_data = state_data[[DIABETES_COL, HOSP_COL]].dropna().copy()
poly_data.columns = ["diabetes", "hosp_stays"]

# Convert diabetes from proportion to percentage for readability.
poly_data["diabetes_pct"] = poly_data["diabetes"] * 100

print(f"Counties with complete data: {len(poly_data)}")
print(f"Preventable stays range: {poly_data['hosp_stays'].min():,.0f} to {poly_data['hosp_stays'].max():,.0f} per 100K")
print(f"Diabetes range: {poly_data['diabetes_pct'].min():.1f}% to {poly_data['diabetes_pct'].max():.1f}%")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(poly_data["hosp_stays"], poly_data["diabetes_pct"],
           alpha=0.7, edgecolors="steelblue", facecolors="lightblue", linewidths=0.8)
ax.set_xlabel("Preventable Hospital Stays (per 100K)", fontsize=12)
ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
ax.set_title("Preventable Hospital Stays vs. Diabetes Prevalence\nCalifornia Counties",
             fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Prepare arrays for scikit-learn.
X_poly = poly_data[["hosp_stays"]].values
y_poly = poly_data["diabetes_pct"].values

# Fit models of increasing complexity.
models_poly = {}
for deg in [1, 2, 3]:
    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    pipe.fit(X_poly, y_poly)
    r2 = pipe.score(X_poly, y_poly)
    models_poly[deg] = pipe
    print(f"Degree {deg}:  R² = {r2:.4f}")

In [ ]:
# Plot the three polynomial fits against the actual data.
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_poly, y_poly, alpha=0.7, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8, label="Actual", zorder=5)

x_smooth = np.linspace(X_poly.min(), X_poly.max(), 200).reshape(-1, 1)
colors_poly = {1: "gray", 2: "orange", 3: "red"}

for deg, pipe in models_poly.items():
    y_hat = pipe.predict(x_smooth)
    r2 = pipe.score(X_poly, y_poly)
    ax.plot(x_smooth, y_hat, color=colors_poly[deg], linewidth=2,
            label=f"Degree {deg} (R²={r2:.3f})")

ax.set_xlabel("Preventable Hospital Stays (per 100K)", fontsize=12)
ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
ax.set_title("Polynomial Regression: Preventable Stays vs. Diabetes", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.5.1 The Danger of High Degree Polynomials

In [ ]:
# Fit high-degree polynomials to show overfitting.
poly_data_over = chr_data[[DIABETES_COL, HOSP_COL]].dropna().copy()
poly_data_over.columns = ["diabetes", "hosp_stays"]
poly_data_over["diabetes_pct"] = poly_data_over["diabetes"] * 100

X_poly = poly_data[["hosp_stays"]].values
y_poly = poly_data["diabetes_pct"].values

overfit_models = {}
for deg in [3, 10, 15, 20]:
    pipe = make_pipeline(PolynomialFeatures(deg), LinearRegression())
    pipe.fit(X_poly, y_poly)
    r2 = pipe.score(X_poly, y_poly)
    overfit_models[deg] = pipe
    print(f"Degree {deg:2d}:  R² = {r2:.4f}")

In [ ]:
# Visualize: R² polynomial degree increases.
fig, ax = plt.subplots(figsize=(9, 6))
degs = [deg for deg,_ in overfit_models.items()]
ax.plot(degs, [pipe.score(X_poly, y_poly) for _,pipe in overfit_models.items()],
        "o-", color="steelblue", linewidth=2, markersize=8, label="R²")

ax.set_xlabel("Polynomial Degree", fontsize=12)
ax.set_ylabel("R²", fontsize=12)
ax.set_title(("Prediction Accuracy: "
              "Preventable Stays vs. " 
              "Diabetes Prevalence"),
             fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xticks(degs)
plt.tight_layout()
plt.show()

#### [BONUS] Plotting the high-degree polynomials

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(X_poly, y_poly, alpha=0.7, edgecolors="steelblue",
           facecolors="lightblue", linewidths=0.8, label="Actual", zorder=5)

styles = {3: ("red", "-"), 10: ("purple", "--"), 15: ("blue", ":"), 20: ("green", ":")}
for deg, pipe in overfit_models.items():
    color, ls = styles[deg]
    y_hat = pipe.predict(x_smooth)
    r2 = pipe.score(X_poly, y_poly)
    ax.plot(x_smooth, y_hat, color=color, linewidth=2, linestyle=ls,
            label=f"Degree {deg} (R²={r2:.3f})")

ax.set_xlabel("Preventable Hospital Stays (per 100K)", fontsize=12)
ax.set_ylabel("Diabetes Prevalence (%)", fontsize=12)
ax.set_title("High-Degree Polynomials Failing", fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3.6 How Models Learn: Teaching a Computer to Get Better Over Time

In [ ]:
# We'll work with the California data from section 3.1.
# Normalize the features so gradient descent converges nicely.
OBESITY_COL = "v011_rawvalue"

X_gd = state_data[OBESITY_COL].values * 100 # percentage
y_gd = state_data[DIABETES_COL].values * 100  # percentage

# Feature scaling: subtract mean, divide by std.
X_mean, X_std = X_gd.mean(), X_gd.std()
y_mean, y_std = y_gd.mean(), y_gd.std()
X_norm = (X_gd - X_mean) / X_std
y_norm = (y_gd - y_mean) / y_std

# Initialize parameters randomly.
np.random.seed(33)
m = np.random.randn()  # slope
b = np.random.randn()  # intercept

# Hyperparameters.
learning_rate = 0.1
n_iterations = 50

# Track the history for visualization.
history = {"iteration": [], "m": [], "b": [], "mse": []}

for i in range(n_iterations):
    # Forward pass: make predictions.
    y_hat = m * X_norm + b

    # Compute loss (MSE).
    error = y_hat - y_norm
    mse = (error ** 2).mean()

    # Compute gradients.
    dm = (2 / len(X_norm)) * (error * X_norm).sum()
    db = (2 / len(X_norm)) * error.sum()

    # Update parameters.
    m -= learning_rate * dm
    b -= learning_rate * db

    # Record history.
    history["iteration"].append(i)
    history["m"].append(m)
    history["b"].append(b)
    history["mse"].append(mse)

print(f"Final MSE (normalized): {history['mse'][-1]:.6f}")
print(f"Converged after {n_iterations} iterations")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
snapshots = [0, 5, n_iterations - 1]

for ax, idx in zip(axes, snapshots):
    ax.scatter(X_norm, y_norm, alpha=0.5, edgecolors="steelblue",
               facecolors="lightblue", linewidths=0.6, s=30)

    m_snap = history["m"][idx]
    b_snap = history["b"][idx]
    x_line = np.array([X_norm.min(), X_norm.max()])
    ax.plot(x_line, m_snap * x_line + b_snap, "r-", linewidth=2)

    ax.set_title(f"Iteration {idx}\nMSE = {history['mse'][idx]:.4f}", fontsize=12)
    ax.set_xlabel("Obesity (normalized)")
    ax.set_ylabel("Diabetes Prevalence (normalized)")
    ax.grid(True, alpha=0.3)

plt.suptitle("Gradient Descent: Watching the Model Learn", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### [BONUS] The Loss Landscape

We can also visualize the loss as a function of the slope and intercept. Gradient descent
walks downhill on this surface toward the minimum.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(history["iteration"], history["mse"], "o-", color="steelblue",
        markersize=4, linewidth=1.5)
ax.set_xlabel("Iteration", fontsize=12)
ax.set_ylabel("Mean Squared Error", fontsize=12)
ax.set_title("Loss Curve: MSE Over Training Iterations", fontsize=14)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.6.1 Comparing to scikit-learn

In [ ]:
# Convert our normalized parameters back to the original scale.
m_original = m * (y_std / X_std)
b_original = y_mean + b * y_std - m_original * X_mean

print("Gradient Descent (our implementation):")
print(f"  Slope:     {m_original:.6f}")
print(f"  Intercept: {b_original:.4f}")
print()
print("scikit-learn (closed-form solution):")
print(f"  Slope:     {model_simple.coef_[0]:.6f}")
print(f"  Intercept: {model_simple.intercept_:.4f}")
print()
diff_m = abs(m_original - model_simple.coef_[0])
diff_b = abs(b_original - model_simple.intercept_)
print(f"Difference in slope:     {diff_m:.8f}")
print(f"Difference in intercept: {diff_b:.8f}")

---

## 3.7 Putting It Together

### 3.7.1 Identifying Priority Counties

In [ ]:
# Refit the multiple regression on all California counties.
model_final = LinearRegression()
model_final.fit(X_multi, y_multi)
predicted = model_final.predict(X_multi)
residuals_final = y_multi - predicted

# Build a summary table.
summary = multi_data[["county"]].copy()
summary.columns = ["County"]
summary["Actual Diabetes (%)"] = y_multi.round(2)
summary["Predicted Diabetes (%)"] = predicted.round(2)
summary["Residual (%)"] = residuals_final.round(2)
summary = summary.sort_values("Residual (%)", ascending=False).reset_index(drop=True)

print("Counties where diabetes is HIGHER than the model predicts")
print("(positive residual = potential priority for intervention)")
print("=" * 65)
print(summary.head(10).to_string(index=False))
print()
print("\nCounties where diabetes is LOWER than the model predicts")
print("(negative residual = potential bright spots to learn from)")
print("=" * 65)
print(summary.tail(10).to_string(index=False))

In [ ]:
# Visualize the residuals on a bar chart.
fig, ax = plt.subplots(figsize=(14, 6))

most_interesting = pd.concat([summary.head(5), summary.tail(5)], ignore_index=True)
colors = ["coral" if r > 0 else "steelblue" for r in most_interesting["Residual (%)"]]
ax.barh(range(len(most_interesting)), most_interesting["Residual (%)"], color=colors, edgecolor="white")
ax.set_yticks(range(len(most_interesting)))
ax.set_yticklabels(most_interesting["County"], fontsize=8)
ax.set_xlabel("Residual: Actual − Predicted Diabetes Prevalence (%)", fontsize=12)
ax.set_title(
    "California Counties: Diabetes Prevalence vs. Model Prediction\n"
    "Coral = higher than expected | Blue = lower than expected",
    fontsize=13,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.grid(True, alpha=0.3, axis="x")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### 3.7.2 What-If Predictions: Simulating an Intervention

In [ ]:
# Simulate a 5-percentage-point reduction in obesity for each county.
predictor_names = list(PREDICTORS.keys())
obesity_idx = predictor_names.index("v011_rawvalue")

X_whatif = X_multi.copy()
X_whatif[:, obesity_idx] = X_whatif[:, obesity_idx] - 0.05  # subtract 5 percentage points

predicted_original = model_final.predict(X_multi)
predicted_whatif = model_final.predict(X_whatif)
predicted_change = predicted_whatif - predicted_original

print("Predicted effect of a 5-percentage-point reduction in obesity rate:")
print(f"  Average change in diabetes prevalence: {predicted_change.mean():.2f} percentage points")
print(f"  Range: {predicted_change.min():.2f} to {predicted_change.max():.2f} percentage points")